In [0]:
## TODO need widgets to pass variables to/from nbs/tasks
# Update paths 

# Data Load & Transforms 

### Image Data Formatting

For object detection and segmentation, image data must be formatted correctly. Two common formats are [COCO](https://cocodataset.org/#home) and [YOLO](https://github.com/AlexeyAB/darknet).

[<img src="./imgs/image_mask_polygon_plots.png" width="600"/>](./imgs/image_mask_polygon_plots.png) 

**Polygons** outline objects in **segmentation masks**, providing: **Simplification** (Reduces mask complexity), **Efficiency** (Less data to process), and **Compatibility** (Required for frameworks like YOLO).
**[COCO Format](https://cocodataset.org/#format-data)** is widely used for its: **Rich Annotations** (Supports bounding boxes, masks, and keypoints), **Standardization** (Well-supported in ML libraries), and **Flexibility** (Suitable for various tasks).
**[YOLO Format](https://github.com/AlexeyAB/darknet)** is designed for YOLO models, offering: **Simplicity** (Minimal, easy-to-parse text format) and **Speed** (Efficient for training and inference).

**Conversion from COCO to YOLO**: **Extract Bounding Boxes** (From COCO masks or polygons) and **Format Annotations** (Convert to YOLO format).

Before processing images in bulk, **visualizing polygons** helps verify accuracy and improve annotations. (ref. image from [Example Image-Mask-Polygon-Viz]($./devs/02_MedCellTypes_data_img2coco_yolo_PLOTs)).

#### YOLO Image Segmentation Format and COCO Format

**YOLO Format**
YOLO (You Only Look Once) is designed for real-time object detection. It uses a simple text-based format:
- **Annotations**: Each line represents an object with class ID, normalized center coordinates (x, y), and normalized width and height.
- **Example**: `class_id x_center y_center width height`

**COCO Format**
COCO (Common Objects in Context) is a comprehensive format for object detection, segmentation, and keypoint detection:
- **Annotations**: Stored in a JSON file with detailed information.
  - **Images**: Metadata about each image.
  - **Annotations**: Includes bounding boxes, segmentation masks (polygons), and keypoints.
  - **Categories**: Class labels and IDs.
- **Example**:
  ```
  json { "images": [{"id": 1, 
                     "file_name": "image1.jpg"
                   }], 
        "annotations": [{"id": 1, 
                         "image_id": 1, "category_id": 1, 
                         "bbox": [x, y, width, height], 
                         "segmentation": [[x1, y1, x2, y2, ...]]
                        }], 
        "categories": [{"id": 1, 
                        "name":"category_name"
                      }] 
      }
  ```

#### Key Differences
- **YOLO**: Simple, efficient for real-time detection, uses bounding boxes.
- **COCO**: Detailed, supports segmentation masks and keypoints, suitable for various tasks.

## Dependencies | Libraries | Paths

In [0]:
!pip install -q ultralytics pycocotools scikit-learn matplotlib # --> clusterlibs or fasterlibloads ?

%restart_python

In [0]:
import os
import shutil

import time
import requests
from zipfile import ZipFile
import json

from pyspark.sql import functions as F, types as T

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import cv2
from PIL import Image
import PIL.Image

from IPython.display import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.utils.data
from torch import nn

# import torchvision
# from torchvision import transforms as T

from pycocotools import mask as coco_mask
from pycocotools.coco import COCO

In [0]:
CATALOG_NAME = "mmt"
SCHEMA_NAME = "cv"

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}"

# DATA_DIR = f"{VOLUME_PATH}/data"
PROJECTS_DIR = f"{VOLUME_PATH}/projects"

PROJECT_PATH = f"{PROJECTS_DIR}/NuInsSeg"

PROJECT_RAWDATA_DIR = f"{PROJECT_PATH}/raw_data"

# ZIPFILE_PATH = f"{PROJECT_RAWDATA_DIR}/nuinsseg.zip"

YOLO_DATA_DIR = f"{PROJECT_PATH}/yolo_dataset" # can update this to change the path to your own data 

# Get the current working directory
nb_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
current_path = f"/Workspace{nb_context}"
# print(f"Current path: {current_path}")
WS_PROJ_DIR = '/'.join(current_path.split('/')[:-1]) 

WORKSPACE_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get("user").get()
USER_WORKSPACE_PATH = f"/Users/{WORKSPACE_PATH}"

# project_name = "yolo_MedCellTypes_InstanceSeg"
# experiment_name = USER_WORKSPACE_PATH + "mlflow_experiments/yolo/{project_name}"
# print(f"Setting experiment name to be {experiment_name}")

--------------------------

## `HELPER Functions`

In [0]:
### IMAGE PROCESSING Helper functions 

# get image and corresponding mask filepaths
def get_image_mask_pairs(data_dir):
    image_paths = []
    mask_paths = []

    for root,_,files in os.walk(data_dir):
        if 'tissue images' in root:
            for file in files:
                if file.endswith('.png'):
                    image_paths.append(os.path.join(root,file))
                    # replace the directory name to 'label masks modify' and the file extension to '.tif'
                    mask_paths.append(os.path.join(root.replace('tissue images','label masks modify'), file.replace('.png','.tif')))
                    
    return image_paths, mask_paths


# convert mask to polygons  
def mask_to_polygons(mask,epsilon=1.0):
    contours,_ = cv2.findContours(mask,cv2.RETR_TREE,cv2.CHAIN_APPROX_SIMPLE) ##accounts for occlusions
    polygons = []
    for contour in contours:
        if len(contour) > 2:
           poly = contour.reshape(-1).tolist()
           if len(poly) > 4: #Ensures valid polygon
              polygons.append(poly)
    return polygons

In [0]:
def create_directories(yolo_data_dir):
    """
    Creates the necessary directories for YOLO dataset if they do not exist.

    Args:
        yolo_data_dir (str): The yolo_data directory where the YOLO dataset directories will be created.
    """
    dirs = {
        'train_images_dir': os.path.join(yolo_data_dir, 'train', 'images'),
        'val_images_dir': os.path.join(yolo_data_dir, 'val', 'images'),
        'test_images_dir': os.path.join(yolo_data_dir, 'test', 'images'),
        'train_labels_dir': os.path.join(yolo_data_dir, 'train', 'labels'),
        'val_labels_dir': os.path.join(yolo_data_dir, 'val', 'labels'),
        'test_labels_dir': os.path.join(yolo_data_dir, 'test', 'labels')
    }

    # Create the output directories if they do not exist
    for dir_path in dirs.values():
        os.makedirs(dir_path, exist_ok=True)
    
    return dirs

--------------------------

## Non-Spark Approach 

### `process_data()` `without` conversion to `pandasUDF`

In [0]:
def process_data(image_paths, mask_paths, output_images_dir, output_labels_dir, processtype_info):
    """
    Processes image and mask data to convert it from COCO format to YOLO format.

    Args:
        image_paths (list): List of paths to the input images.
        mask_paths (list): List of paths to the corresponding mask images.
        output_images_dir (str): Directory to save the processed images.
        output_labels_dir (str): Directory to save the YOLO format labels.

    Returns:
        coco_input, images, annotations
    """

    print(f"PROCESSING --- {processtype_info} --- SPLIT")
    print("-"*100)
    
    # Initialize Lists and Counters: Initializes empty lists for annotations and images, and counters for image_id and ann_id.
    annotations = []
    images = []
    image_id = 0
    ann_id = 0

    # Iterate Over Image and Mask Paths: Loops through the provided image and mask paths.
    print("Read image and mask as cv2 files... ")
    for img_path, mask_path in zip(image_paths, mask_paths):
        image_id += 1
        # Read and Copy Images: Reads each image and mask using OpenCV, and copies the image to the specified output directory.
        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        print("Copy image path to output directory")
        shutil.copy(img_path, os.path.join(output_images_dir, os.path.basename(img_path)))

        # Store Image Metadata: Adds metadata for each image (ID, file name, height, width) to the images list.
        print("Append image metadata to images list")
        images.append({
            "id": image_id,
            "file_name": os.path.basename(img_path),
            "height": img.shape[0],
            "width": img.shape[1]
        })

        # Process Masks: Identifies unique values in the mask to distinguish different objects. 
        print("Process masks...")
        unique_values = np.unique(mask)
        for value in unique_values:
            if value == 0:  # Ignore background
                continue

            # For each unique value (excluding the background), it creates a binary mask and converts it to polygons.
            print("Convert mask to polygons")
            object_mask = (mask == value).astype(np.uint8) * 255
            polygons = mask_to_polygons(object_mask)

            # Store Annotations: For each polygon, it creates an annotation with the image ID, category ID, and segmentation data, and adds it to the annotations list.
            print("Append polygon as segmentation with image_id & category_id to annotations")
            for poly in polygons:
                ann_id += 1
                annotations.append({
                    "image_id": image_id,
                    "category_id": 1,  # Only one category: Nuclei 
                    "segmentation": [poly],
                })

    # Create COCO-like Dictionary: Constructs a dictionary in COCO format containing the images, annotations, and categories.
    print("Create COCO-like Dictionary...")
    coco_input = {
        "images": images,
        "annotations": annotations,
        "categories": [{"id": 1, "name": "Nuclei"}]
    }

    # Convert to YOLO Format: Converts the COCO-like annotations to YOLO format by normalizing the polygon coordinates and writing them to text files in the specified output directory.
    print("Convert to YOLO Format...")
    for img_info in coco_input["images"]:
        img_id = img_info["id"]
        img_ann = [ann for ann in coco_input["annotations"] if ann["image_id"] == img_id]
        img_w, img_h = img_info["width"], img_info["height"]
        
        if img_ann:
            print(f"Write annotations | normalized polygon(s) to {img_info['file_name']} ...")
            with open(os.path.join(output_labels_dir, os.path.splitext(img_info["file_name"])[0] + '.txt'), 'w') as file_object:
                for ann in img_ann:
                    current_category = ann['category_id'] - 1
                    polygon = ann['segmentation'][0]
                    # print(f"Writing normalized polygon to {img_info['file_name']}")
                    normalized_polygon = [format(coord / img_w if i % 2 == 0 else coord / img_h, '.6f') for i, coord in enumerate(polygon)]
                    file_object.write(f"{current_category} " + " ".join(normalized_polygon) + "\n")
    
    return coco_input, images, annotations

###  `yolo_dataset_preparation()`

In [0]:
def yolo_dataset_preparation():
    
    # Create the necessary directories
    print('Creating directories...')
    dirs = create_directories(YOLO_DATA_DIR)

    # Ensure directories are created
    for dir_path in dirs.values():
        while not os.path.exists(dir_path):
            print(f"Waiting for directory {dir_path} to be created...")
            time.sleep(1)

    # Short pause
    time.sleep(30)

    # Get image and mask paths
    print('Getting image and mask paths...')
    image_paths, mask_paths = get_image_mask_pairs(PROJECT_RAWDATA_DIR) 

    # Split data into train, val, and test
    print('Splitting data into train, val, and test...')
    train_img_paths, temp_img_paths, train_mask_paths, temp_mask_paths = train_test_split(
        image_paths, mask_paths, test_size=0.4, random_state=42) ## update if you change the split ratio / random_state
    val_img_paths, test_img_paths, val_mask_paths, test_mask_paths = train_test_split(
        temp_img_paths, temp_mask_paths, test_size=0.5, random_state=42) ## update if you change the split ratio / random_state

    # Define the input data for process_data function
    print('Defining inputs for data processing...')
    train_val_test = [
        (train_img_paths, train_mask_paths, dirs["train_images_dir"], dirs["train_labels_dir"], "train"),
        (val_img_paths, val_mask_paths, dirs["val_images_dir"], dirs["val_labels_dir"], "val"),
        (test_img_paths, test_mask_paths, dirs["test_images_dir"], dirs["test_labels_dir"], "test")
    ]

    # Use list comprehension to call process_data for each data split    
    print('Processing data...')
    results = [
                process_data(*data_split) for data_split in train_val_test
                if all(os.path.exists(path) for path in data_split[0])
              ]

    # Unpack the results
    print('Unpacking results... to return')
    (coco_input_train, images_train, annotations_train), (coco_input_val, images_val, annotations_val), (coco_input_test, images_test, annotations_test) = results

    return coco_input_train, images_train, annotations_train, coco_input_val, images_val, annotations_val, coco_input_test, images_test, annotations_test

### Non-Distributed Image & Mask Data Processing 
Derive COCO--YOLO format for data & annotations

In [0]:
coco_input_train, images_train, annotations_train, coco_input_val, images_val, annotations_val, coco_input_test, images_test, annotations_test = yolo_dataset_preparation() 

# 4-8mins 

--------------------------

In [0]:
# coco_input_test.keys()

In [0]:
# coco_input_test['images']

In [0]:
# coco_input_test['annotations'][:10]

--------------------------

## Spark Approach: Distributing `Train-Val-Test` Image Data Processing 

### `process_data_udf()` `WITH` conversion to `pandasUDF`

In [0]:
# Processes image and mask data to convert it from COCO format to YOLO format.

@F.pandas_udf("string", F.PandasUDFType.SCALAR)
def process_data_udf(image_paths: pd.Series, mask_paths: pd.Series, output_images_dir: pd.Series, output_labels_dir: pd.Series, processtype_info: pd.Series) -> pd.Series:
    
    inputs = []
    
    for img_path, mask_path, out_img_dir, out_lbl_dir in zip(image_paths, mask_paths, output_images_dir, output_labels_dir):
        # Ensure the input is a string
        img_path = str(img_path)
        mask_path = str(mask_path)
        out_img_dir = str(out_img_dir)
        out_lbl_dir = str(out_lbl_dir)

        # Read and Copy Images: Reads each image and mask using OpenCV, and copies the image to the specified output directory.
        print("Read image and mask as cv2 files... ")
        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        print("Copy image path to output directory")
        shutil.copy(img_path, os.path.join(out_img_dir, os.path.basename(img_path)))

        # Initialize Lists and Counters: Initializes empty lists for annotations and images, and counters for image_id and ann_id.
        images = []
        annotations = []
        image_id = 0
        ann_id = 0

        # Store Image Metadata: Adds metadata for each image (ID, file name, height, width) to the images list.
        print("Append image metadata to images list")
        images.append({
            "id": image_id,
            "file_name": os.path.basename(img_path),
            "height": img.shape[0],
            "width": img.shape[1]
        })

        # Process Masks: Identifies unique values in the mask to distinguish different objects. 
        print("Process masks...")
        unique_values = np.unique(mask)
        for value in unique_values:
            if value == 0:
                continue

            # For each unique value (excluding the background), it creates a binary mask and converts it to polygons.
            print("Convert mask to polygons")
            object_mask = (mask == value).astype(np.uint8) * 255
            polygons = mask_to_polygons(object_mask)

            # Store Annotations: For each polygon, it creates an annotation with the image ID, category ID, and segmentation data, and adds it to the annotations list.
            print("Append polygon as segmentation with image_id & category_id to annotations")
            for poly in polygons:
                ann_id += 1
                annotations.append({
                    "image_id": image_id,
                    "category_id": 1,
                    "segmentation": [poly],
                })

        # Create COCO-like Dictionary: Constructs a dictionary in COCO format containing the images, annotations, and categories.
        print("Create COCO-like Dictionary...")
        coco_input = {
            "images": images,
            "annotations": annotations,
            "categories": [{"id": 1, "name": "Nuclei"}]
        }

        # Convert to YOLO Format: Converts the COCO-like annotations to YOLO format by normalizing the polygon coordinates and writing them to text files in the specified output directory.
        print("Convert to YOLO Format...")
        for img_info in coco_input["images"]:
            img_id = img_info["id"]
            img_ann = [ann for ann in coco_input["annotations"] if ann["image_id"] == img_id]
            img_w, img_h = img_info["width"], img_info["height"]

            if img_ann:
                print(f"Write annotations | normalized polygon(s) to {img_info['file_name']} ...")
                with open(os.path.join(out_lbl_dir, os.path.splitext(img_info["file_name"])[0] + '.txt'), 'w') as file_object:
                    for ann in img_ann:
                        current_category = ann['category_id'] - 1
                        polygon = ann['segmentation'][0]
                        # print(f"Writing normalized polygon to {img_info['file_name']}")
                        normalized_polygon = [format(coord / img_w if i % 2 == 0 else coord / img_h, '.6f') for i, coord in enumerate(polygon)]
                        file_object.write(f"{current_category} " + " ".join(normalized_polygon) + "\n")
    
        # Append results as JSON strings
        inputs.append(json.dumps({
                                    "coco_input": coco_input,
                                    "images": images,
                                    "annotations": annotations
                                  }
                                )
                      )

    return pd.Series(inputs)

### `yolo_dataset_preparation_sparkpandasUDF()`

In [0]:
# Define the schema for the expected output
inputs_schema = T.StructType([
    T.StructField("coco_input", T.StructType([
        T.StructField("images", T.ArrayType(T.StructType([
            T.StructField("id", T.IntegerType(), True),
            T.StructField("file_name", T.StringType(), True),
            T.StructField("height", T.IntegerType(), True),
            T.StructField("width", T.IntegerType(), True)
        ])), True),
        T.StructField("annotations", T.ArrayType(T.StructType([
            T.StructField("image_id", T.IntegerType(), True),
            T.StructField("category_id", T.IntegerType(), True),
            T.StructField("segmentation", T.ArrayType(T.ArrayType(T.FloatType())), True)
        ])), True),
        T.StructField("categories", T.ArrayType(T.StructType([
            T.StructField("id", T.IntegerType(), True),
            T.StructField("name", T.StringType(), True)
        ])), True)
    ]), True),
    T.StructField("images", T.ArrayType(T.StructType([
        T.StructField("id", T.IntegerType(), True),
        T.StructField("file_name", T.StringType(), True),
        T.StructField("height", T.IntegerType(), True),
        T.StructField("width", T.IntegerType(), True)
    ])), True),
    T.StructField("annotations", T.ArrayType(T.StructType([
        T.StructField("image_id", T.IntegerType(), True),
        T.StructField("category_id", T.IntegerType(), True),
        T.StructField("segmentation", T.ArrayType(T.ArrayType(T.FloatType())), True)
    ])), True)
])

In [0]:
# Apply the schema to the inputs column
def yolo_dataset_preparation_sparkpandasUDF():
    # Create the necessary directories
    print('Creating directories...')
    dirs = create_directories(YOLO_DATA_DIR)

    # Ensure directories are created
    for dir_path in dirs.values():
        while not os.path.exists(dir_path):
            print(f"Waiting for directory {dir_path} to be created...")
            time.sleep(1)

    ## Short pause
    # time.sleep(10)

    # Get image and mask paths
    print('Getting image and mask paths...')
    image_paths, mask_paths = get_image_mask_pairs(PROJECT_RAWDATA_DIR) #(DATA_PROJECT_DIR) 

    # Split data into train, val, and test || update to change % of data to be used
    print('Splitting data into train, val, {and test}...') # 1330 files in total 
    
    ## train-val-test split
    train_img_paths, temp_img_paths, train_mask_paths, temp_mask_paths = train_test_split(
        image_paths, mask_paths, test_size=0.4, random_state=42)
    val_img_paths, test_img_paths, val_mask_paths, test_mask_paths = train_test_split(
        temp_img_paths, temp_mask_paths, test_size=0.5, random_state=42)

    
    # Define the input data for process_data function
    print('Defining inputs for data processing...')
    train_val_test = [
        (train_img_paths, train_mask_paths, dirs["train_images_dir"], dirs["train_labels_dir"], "train"),
        (val_img_paths, val_mask_paths, dirs["val_images_dir"], dirs["val_labels_dir"], "val"),
        (test_img_paths, test_mask_paths, dirs["test_images_dir"], dirs["test_labels_dir"], "test")
    ]
        

    # Initialize an empty DataFrame
    combined_df = None

    # Convert lists to Spark DataFrames and process them
    print('Data Splits Processing with pandasUDF...')
    for idx, (img_paths, mask_paths, output_images_dir, output_labels_dir, processtype_info) in enumerate(train_val_test):
        print(f"{idx}: Processing {processtype_info} data...")
        df = spark.createDataFrame(pd.DataFrame({"image_paths": img_paths, "mask_paths": mask_paths}))
        df = df.withColumn("output_images_dir", F.lit(output_images_dir))
        df = df.withColumn("output_labels_dir", F.lit(output_labels_dir))
        df = df.withColumn("processtype_info", F.lit(processtype_info))
        
        # Apply the pandasUDF to process_data()
        df = df.withColumn("inputs", process_data_udf(df["image_paths"], df["mask_paths"], df["output_images_dir"], df["output_labels_dir"], df["processtype_info"]))
        
        # Cast the inputs column to the defined schema
        df = df.withColumn("inputs", F.from_json(df["inputs"], inputs_schema)) ##
        
        # Append the DataFrame to the combined DataFrame
        print(f"Appending {processtype_info} data to combined DataFrame...")
        if combined_df is None:
            combined_df = df
        else:
            combined_df = combined_df.union(df)

    print('-------DONE!------')
    return combined_df

--------------------------

### Distributed Image & Mask Data Processing 
Derive COCO--YOLO format for data & annotations with vectorized `process_data_udf()`

In [0]:
process_img_df = yolo_dataset_preparation_sparkpandasUDF() 

# 46s ~1min

### Break down of Image Data and Masks for Data Splits 

In [0]:
from pyspark.sql import functions as F

# Group by 'processtype_info' and count the occurrences
grouped_df = process_img_df.groupby("processtype_info").count()

# Calculate the total count
total_count = grouped_df.agg(F.sum("count")).collect()[0][0]

# Add a new column for the percentage
grouped_df = grouped_df.withColumn(
    "percentage", 
    (F.col("count") / F.lit(total_count)) * 100
)

# Display the DataFrame with counts and percentages
display(grouped_df)

### Processing paths and inputs corresponding to `Train-Val-{Test}` Data Splits  

In [0]:
display(process_img_df.sort('image_paths'))

### EXTRACT `inputs` info. as Columns 

In [0]:
process_img_df2 = process_img_df.select("*", "inputs.*").sort('image_paths')

display(process_img_df2)

--------------------------

### WRITE OUT `processing paths` and `inputs` as reference table in UC 

In [0]:
tablename2use = f"{CATALOG_NAME}.{SCHEMA_NAME}.yolo_nuinsseg_dataset_train60val20test20"
print(tablename2use) 
process_img_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{tablename2use}")

--------------------------